In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo 

server_timestamp = datetime.now(ZoneInfo("New_York")).strftime("%Y-%m-%d %H:%M:%S")
print(server_timestamp)

In [ ]:
# Generating new table in sql db
import sqlite3

conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS workouts (
        work_list TEXT)''')
conn.commit()
conn.close()

In [ ]:
import json
import sqlite3
# adding a list of workouts to the table
conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
# intial workout list gained from original google sheets file
workouts = [
    "Biceps seated db curls",
    "Back row machine",
    "Core abdominal machine",
    "Core torso rotation machine",
    "Biceps curl machine",
    "Back ISO rear deltoid machine",
    "Chest press machine",
    "Triceps extension machine",
    "Shoulders db complex",
    "Chest pectoral fly machine",
    "Triceps db skull crushers",
    "Back pull-down machine",
    "Biceps db preacher curls",
    "Chest db bench press",
    "Leg extension machine",
    "Leg curl machine",
    "Hip abduction",
    "Hip adduction",
    "Calf leg press machine",
    "Leg press machine",
    "Biceps standing db curls",
    "Triceps pull-down",
    "Biceps curl machine high",
    "Back rear deltoid machine",
    "Chest smith machine bench",
    "Chest incline press machine",
    "Back extension machine",
    "Core complex",
    "Shoulders press machine",
    "Triceps press machine",
    "Chest push ups",
    "Hip addiction machine",
    "Calf extension machine",
    "Chest bench press",
    "Back assisted pull up machine",
    "Shoulders deltoid raise machine",
    "Trcieps pull-down",
    "other"]

workouts_json = json.dumps(workouts)

cursor.execute('''
            INSERT INTO workouts (work_list)
            VALUES (?)
        ''', (workouts_json,))
conn.commit()
conn.close()

In [ ]:
# Reading in the json from the sql db
import json
import sqlite3

conn = sqlite3.connect("habits.db")
cursor = conn.cursor()
cursor.execute("SELECT work_list FROM workouts")
result = cursor.fetchone() # only fetches one row from db

workout_list = json.loads(result[0])
print(workout_list)  # results back into list

['Bench Press', 'Squats', 'Deadlifts', 'Overhead Press', 'Pull-Ups', 'Push-Ups', 'Lunges', 'Plank', 'Burpees', 'Bicep Curls']


In [1]:
##### WHEN READING IN UPDATED DATA FROM GOOGLE SHEET MAKE SURE TO RUN SQL COMMAND TO TRIM WHITESPACES FROM EXERCISE COLUMN ######

# Merge exisiting workouts google sheet to sql db

import pandas as pd
from sqlalchemy import create_engine
import pandas as pd
import os

sheet_id = "1TM8IcrqlYRlk8Ggxig1XMGh5e12KzZTGbh41tp6IB6k"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

workouts = pd.read_csv(sheet_url)

# Ensure timestamp is in datetime format
workouts['Timestamp'] = pd.to_datetime(workouts['Timestamp'])

def df_to_sqlite(df: pd.DataFrame, name: str, db_path: str) -> None:
    # Input validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not isinstance(db_path, str):
        raise TypeError("db_path must be a string")

    # Create a SQLite connection string
    engine = create_engine(f"sqlite:///{db_path}")

    # Save the DataFrame to the SQLite database
    df.to_sql(name=name, con=engine, if_exists='replace', index=True)

    # Dispose the engine
    engine.dispose()


df_to_sqlite(workouts, "workouts", "habits.db")

In [ ]:
# Getting all the unqiue book titles from the habit_logs table
import json
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///habits.db")

# read_sql_query simple 
with engine.connect() as connection:
    df_filter = pd.read_sql_query(
        text("""SELECT * FROM habit_logs
             WHERE habit_type = 'reading' and 'workout'"""), connection)

engine.dispose()

# Parse JSON strings into dictionaries
df_parsed = df_filter["data"].apply(json.loads)

# Expand JSON dicts into separate columns
df_expanded = pd.json_normalize(df_parsed)

book_titles = df_expanded["book_title"].unique().tolist()

In [ ]:
# Getting all the unqiue book titles from the habit_logs table
import json
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///habits.db")

# read_sql_query simple 
with engine.connect() as connection:
    workouts = pd.read_sql_query(
        text("""SELECT exercise FROM workouts"""), connection)

engine.dispose()

workouts_list = sorted(workouts["Exercise"].unique().tolist())

In [ ]:
# Cleaning the sql db and saving it back to the same table, specifically clearing leading/trailing whitespace from the 'exercise' column

import pandas as pd
from sqlalchemy import create_engine, text

# Connect to your database
engine = create_engine("sqlite:///habits.db")  # or other dialects like postgresql://...

# Step 1: Read the table into pandas
with engine.connect() as connection:
    df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

# Step 2: Clean the data (strip leading/trailing whitespace from 'exercise' column)
df['Exercise'] = df['Exercise'].str.strip()

# Step 4: Save it back to the same table (overwrite or update)
df.to_sql("workouts", engine, if_exists="replace", index=False)

234

In [ ]:
import pandas as pd
from sqlalchemy import create_engine,text
import pandas as pd
import json

# Initializing the SQLite database link for the entire app
DATABASE = 'habits.db'

engine = create_engine(f"sqlite:///{DATABASE}")

# read_sql_query simple 
with engine.connect() as connection:
    whole_table = pd.read_sql_query(
        text("""SELECT * FROM habit_logs
            WHERE habit_type = 'reading'"""), connection)

# Parse JSON strings into dictionaries
df_parsed = whole_table["data"].apply(json.loads)

# Expand JSON dicts into separate columns
df_expanded = pd.json_normalize(df_parsed)

# Grtabbing unique book titles from the df and sorting them
book_titles = sorted(df_expanded["book_title"].unique().tolist())

# Add "Other" option to the list of book titles and works. This way other is not saved to the database
book_titles.append("Other")
        

#### Exmaples of graphs and statistics I want to show on the static visualization dashbaord

In [ ]:
# YTD Workouts w/ tagerts
# Grabbing monthly exercise and workout statistics for this month and last month to be displayed
import pandas as pd
from sqlalchemy import create_engine, text
import datetime as dt

def get_kpi_stats():

    # Initializing the SQLite database link
    DATABASE = 'habits.db'
    engine = create_engine(f"sqlite:///{DATABASE}")

    # Read workouts data from database
    with engine.connect() as connection:
        workout_df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

    # Ensure timestamp is in datetime format
    workout_df['Timestamp'] = pd.to_datetime(workout_df['Timestamp'])

    workout_df['month'] = workout_df['Timestamp'].dt.month_name()
    workout_df['year'] = workout_df['Timestamp'].dt.year
    workout_df['date'] = workout_df['Timestamp'].dt.date

    # Get current date and year/month details
    current_date = pd.Timestamp.now()
    current_year = current_date.year
    current_month = current_date.month

    # Calculate Year-to-Date count
    ytd_count = len(workout_df[workout_df['year'] == current_year])

    # Calculate Month-to-Date count
    mtd_count = len(workout_df[(workout_df['year'] == current_year) & 
                        (workout_df['month'] == current_date.month_name())])

    # You can also calculate previous month's total for comparison
    prev_month = current_month - 1 if current_month > 1 else 12
    year = current_year if current_month > 1 else current_year - 1

    prev_month_count = len(workout_df[(workout_df['Timestamp'].dt.year == year) & 
                                    (workout_df['Timestamp'].dt.month == prev_month)])

    # MTD Workouts w/ tagerts

    # Excersies per workout avg

    # days exercised by month
    workouts_by_month_df = workout_df.groupby(['month', 'year']).agg({'Timestamp': 'count'})

    # Days exercised by month
    days_per_month_df = workout_df.groupby(['date', 'month']).agg({'Timestamp': 'count'}).groupby(['month']).agg({'Timestamp': 'count'})

    # Format your DataFrame tables as HTML with styling
    days_per_month_html = days_per_month_df.to_html(classes='workout-table', index=True, border=0)
    workouts_by_month_html = workouts_by_month_df.to_html(classes='workout-table', index=True, border=0)

    return ytd_count, mtd_count, prev_month_count, days_per_month_html, workouts_by_month_html

ytd_count, mtd_count, prev_month_count, days_per_month_html, workouts_by_month_html = get_kpi_stats()

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text
import datetime as dt

# Initializing the SQLite database link
DATABASE = 'habits.db'
engine = create_engine(f"sqlite:///{DATABASE}")

# Read workouts data from database
with engine.connect() as connection:
    workout_df = pd.read_sql_query(text("SELECT * FROM workouts"), connection)

# Ensure timestamp is in datetime format
workout_df['Timestamp'] = pd.to_datetime(workout_df['Timestamp'])

workout_df['month'] = workout_df['Timestamp'].dt.month_name()
workout_df['year'] = workout_df['Timestamp'].dt.year
workout_df['date'] = workout_df['Timestamp'].dt.date

# Get current date and year/month details
current_date = pd.Timestamp.now()
current_year = current_date.year
current_month = current_date.month

# Calculate Year-to-Date count
ytd_count = len(workout_df[workout_df['year'] == current_year])

# Calculate Month-to-Date count
mtd_count = len(workout_df[(workout_df['year'] == current_year) & 
                        (workout_df['month'] == current_date.month_name())])


# Trying to import Apple's health data from the xml 

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load and parse the file
tree = ET.parse('apple/export.xml') 
root = tree.getroot()

# Extract data from <Record> tags
records = []
for record in root.findall('Record'):
    records.append(record.attrib)

# Convert to DataFrame
df = pd.DataFrame(records)

#df = df[df["sourceName"] == "Ryan’s Apple Watch"]

In [ ]:
(df["type"]).unique()

In [6]:
filtered_df = df[df["type"] == "HKQuantityTypeIdentifierPhysicalEffort"]